# 🔍 PaddleOCR Document Pipeline - Colab

PP-StructureV3 + PP-OCRv5 기반 문서 파싱 서비스

**사전 설정**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택

## 1. 레포 클론 & 의존성 설치

In [ ]:
# 레포 클론
!git clone -b simple-test https://github.com/dongtan-91-dong-welfare-center/audit_inquiry_automation.git /content/app
%cd /content/app

In [ ]:
# PaddlePaddle GPU 설치 (Colab T4 = CUDA 12.x)
!pip install -q paddlepaddle-gpu==3.1.0

# PaddleOCR + paddlex[ocr] + 서버 의존성
!pip install -q "paddleocr>=3.1.0" "paddlex[ocr]" \
    fastapi uvicorn python-multipart pydantic pydantic-settings \
    aiofiles PyMuPDF pyngrok nest-asyncio

In [ ]:
# 설치 확인
import paddle
print(f"PaddlePaddle: {paddle.__version__}")
print(f"GPU available: {paddle.device.is_compiled_with_cuda()}")
print(f"GPU: {paddle.device.get_device()}")

## 2. 환경 설정

In [ ]:
import os

# 모델 소스 체크 비활성화 (속도 향상)
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

# 모델 캐시 디렉토리 (Colab은 권한 문제 없음)
os.environ.setdefault("PADDLEX_MODEL_DIR", "/content/models")
os.makedirs("/content/models", exist_ok=True)

print("✅ 환경 설정 완료")

## 3. 서버 실행 (2가지 방식 중 선택)

- **방식 A**: ngrok 터널 → 외부 URL 생성 (브라우저에서 UI 접근 가능)
- **방식 B**: Colab 내부에서만 사용 (노트북 셀에서 API 호출)

### 방식 A: ngrok 터널 (외부 접속 가능)

In [ ]:
import sys
import threading
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

# 앱 모듈 경로 추가
sys.path.insert(0, "/content/app")
os.chdir("/content/app")

from app import app

# ngrok 터널 생성
# 무료 계정: https://dashboard.ngrok.com/get-started/your-authtoken
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")  # 선택사항 - 없어도 동작

public_url = ngrok.connect(8090)
print(f"")
print(f"========================================")
print(f"🌐 외부 접속 URL: {public_url}")
print(f"📄 Swagger UI:   {public_url}/docs")
print(f"🖥️  Web UI:       {public_url}/")
print(f"========================================")
print(f"")

# 백그라운드 스레드로 서버 실행
thread = threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "0.0.0.0", "port": 8090, "log_level": "info"},
    daemon=True,
)
thread.start()
print("✅ 서버 시작됨 (첫 요청 시 모델 로딩 ~30초)")

### 방식 B: ngrok 없이 내부 전용

In [ ]:
# # 방식 A를 실행했으면 이 셀은 건너뛰세요
# import sys
# import threading
# import nest_asyncio
# import uvicorn
#
# nest_asyncio.apply()
# sys.path.insert(0, "/content/app")
# os.chdir("/content/app")
#
# from app import app
#
# thread = threading.Thread(
#     target=uvicorn.run,
#     args=(app,),
#     kwargs={"host": "0.0.0.0", "port": 8090, "log_level": "info"},
#     daemon=True,
# )
# thread.start()
# BASE_URL = "http://localhost:8090"
# print(f"✅ 내부 서버: {BASE_URL}")

## 4. 테스트

In [ ]:
import requests
import time

BASE_URL = "http://localhost:8090"  # 내부 호출은 항상 localhost

# 헬스체크
time.sleep(3)
resp = requests.get(f"{BASE_URL}/health")
print("Health:", resp.json())

In [ ]:
# 테스트 파일 업로드
# Colab에 파일 업로드하거나 URL로 다운로드
from google.colab import files

uploaded = files.upload()  # 파일 선택 대화상자
test_file = list(uploaded.keys())[0]
print(f"업로드된 파일: {test_file}")

In [ ]:
# PP-StructureV3 모드로 파싱
import json

with open(test_file, "rb") as f:
    resp = requests.post(
        f"{BASE_URL}/parse",
        files={"file": (test_file, f)},
        data={"mode": "structure"},
        timeout=300,
    )

result = resp.json()
print(f"✅ 파싱 완료: {result['filename']}")
print(f"   총 페이지: {result['total_pages']}")
print(f"   Markdown 길이: {len(result.get('full_markdown', ''))} chars")
print()
print("=== Markdown 미리보기 (앞 1000자) ===")
print(result.get("full_markdown", "")[:1000])

In [ ]:
# Markdown 전체 저장 & 다운로드
md_filename = test_file.rsplit(".", 1)[0] + ".md"
with open(md_filename, "w") as f:
    f.write(result.get("full_markdown", ""))

# JSON 전체 저장
json_filename = test_file.rsplit(".", 1)[0] + ".json"
with open(json_filename, "w") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"📥 {md_filename}, {json_filename} 저장됨")
files.download(md_filename)
files.download(json_filename)

## 5. (선택) 서버 종료 & 정리

In [ ]:
# ngrok 터널 종료
try:
    ngrok.kill()
    print("ngrok 종료됨")
except:
    pass